Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 150

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import openaire_doi_get_info, _opeanaire_get_subjects

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2304061447', 'title': 'Munich Crystallography BAG', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2304061447</identifier>\n    <datestamp>2026-01-29T07:00:09.355Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2304061447</identifier><creators><creator><creatorName nameType="Personal">Andre MOURAO</c

In [7]:
record_titles

['ITABAG: 3D structures of macromolecules related to human health',
 'Structure of metalloproteins, membrane proteins, filamentous sarcomer proteins, and virus particles',
 'PROTEINS RELATED TO PATHOGENESIS OF DISEASES, VIRAL PROTEINS, CELL DIVISION, SIGNALLING AND CHROMATIN PROCESSES',
 'Dynamics in hydrogenated metallic glasses',
 'Growth and structure of thin CuO films in epitaxy on STO(001)',
 'Analysis of structure and stress- induced Liquid-Liquid Phase Separation of plant-specific splicing factor',
 'IHR',
 'Orientation resolved local strain in in-situ compressed cortical bone',
 'Structure and reactivity of As under extreme conditions: high pressure and high temperature synthesis of H-rich extended As-H structures',
 'Graphene growth on liquid metals. Continuation of running experiment MA-5338 with the same safety requirements.',
 'Macromolecular Crystallography at South-East Andalusia',
 'Nanoimaging of various biological and materials science samples',
 'Characterizing the ch

In [8]:
record_raw_metadata[10]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/686525006</identifier>\n    <datestamp>2022-05-08T06:00:13.122Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-686525006</identifier><creators><creator><creatorName nameType="Personal">Jose Antonio GAVIRA GALLARDO</creatorName><givenName>Jose A.</givenName><familyName>Gavira Gallardo</familyName><nameIdentifier nameIden

In [8]:
def ET_from_string(string):
    return ET.ElementTree(ET.fromstring(string))

In [10]:
raw_xml = ET_from_string(record_raw_metadata[13])

In [11]:
raw_xml.getroot()

<Element {http://www.openarchives.org/OAI/2.0/}record at 0x22a383ebec0>

In [12]:
#print(ET.tostring(raw_xml, pretty_print=True).decode())

In [13]:
# Finding a desired tag anywhere in tree, e.g. title
raw_xml.xpath("//*[local-name()='title']/text()")

['Probing the protein corona around colloidal metal nanoparticles in cellular environments using nanofocused X-ray synchrotron radiation']

In [14]:
element  = raw_xml.findall(".//{*}title")
print(element)



print(element[0].text)

[<Element {http://schema.datacite.org/meta/kernel-4.2/}title at 0x22a3840e6c0>]
Probing the protein corona around colloidal metal nanoparticles in cellular environments using nanofocused X-ray synchrotron radiation


In [15]:
element  = raw_xml.findall(".//{*}identifier[@identifierType='DOI']")
element
#element[0].text

[<Element {http://schema.datacite.org/meta/kernel-4.2/}identifier at 0x22a3840f6c0>]

In [16]:
raw_xml.xpath("//*[local-name()='date']/text()")

['2021-07-02T06:00:00Z/2021-07-08T06:00:00Z', '2021-07-08T06:00:00Z']

In [17]:
# getting the DOI of the dataset
# Later, find related DOIs, articles...

# xpath returns a list of strings if /text() is included

raw_xml.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

['10.15151/ESRF-ES-450250995']

In [18]:
record_number = 6

print(ET.tostring(ET_from_string(record_raw_metadata[record_number - 1]), pretty_print=True).decode())

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <header>
    <identifier>oai:icatplus.esrf.fr:inv/2234398948</identifier>
    <datestamp>2025-10-02T06:00:09.756Z</datestamp>
  </header>
  <metadata>
    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">
      <schemaVersion>4.2</schemaVersion>
      <datacentreSymbol>EI</datacentreSymbol>
      <payload>
        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2234398948</identifier><creators><creator><creatorName nameType="Personal">Marta GAPINSKA</creatorName><givenName>Marta</givenName><familyName>Gapinska</familyName><nameIdentifier nameIdentifierScheme="ORCID">0000-0001-5

In [19]:
rec = ET_from_string(record_raw_metadata[5])

In [20]:
rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")[0]

'10.15151/ESRF-ES-2234398948'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [9]:
rec = ET_from_string(record_raw_metadata[9])
doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

if doi == []:
    print("NO DOI")
else:
    print(openaire_doi_get_info(doi[0]))

DEBUG:    2026-05-21 15:12:32 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 15:12:32 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-2113944572 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Morris, Liz', 'name': 'Liz', 'surname': 'Morris', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-1893-7515'}, 'provenance': None}}, {'fullName': 'Purkiss, Andrew', 'name': 'Andrew', 'surname': 'Purkiss', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-5931-3509'}, 'provenance': None}}, {'fullName': 'Willoughby, Jemima', 'name': 'Jemima', 'surname': 'Willoughby', 'rank': 3, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'BM29'}, 'provenance': None}, {'subject': {'scheme': 'ke

In [10]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [11]:
import random
import time

results = []
for i in ix_has_doi:
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

    if doi == []:
        print(f"{i} : NO DOI")
    else:
        doi = doi[0]

        # small pause to not overwhelm API
        time.sleep(random.uniform(0.01, 0.5))

        print(f"DOI for record {i}")
        try:
            res = openaire_doi_get_info(doi)
        except Exception as e:
            print(f"ERROR: Record {i} met with error: {str(e)}")
            res = {}

        print(res)
        results.append(res)

DOI for record 0
DEBUG:    2026-05-21 15:13:15 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 15:13:16 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-2304061447 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Tulika CHAKRABORTTY', 'name': 'Tulika', 'surname': 'Chakrabortty', 'rank': 1, 'pid': None}, {'fullName': 'Andre MOURAO', 'name': 'Andre', 'surname': 'Mourao', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-0764-9868'}, 'provenance': None}}, {'fullName': 'Vasilii MOROZOV', 'name': 'Vasilii', 'surname': 'Morozov', 'rank': 3, 'pid': None}, {'fullName': 'Andreas BRACHER', 'name': 'Andreas', 'surname': 'Bracher', 'rank': 4, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-8530-7594'}, 'provenance': None}}, {'fullName': 'Sabine SCHNEIDER', 'name': 'Sabine', 'surname': 'Schneider', 'rank': 5, 'pid': {'id': {'scheme': 'orcid_pending',

In [25]:
results[6]

[{'authors': [{'fullName': 'Alice LLEWELLYN',
    'name': 'Alice',
    'surname': 'Llewellyn',
    'rank': 1,
    'pid': None},
   {'fullName': 'Partha PAUL',
    'name': 'Partha',
    'surname': 'Paul',
    'rank': 2,
    'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-0559-3429'},
     'provenance': None}},
   {'fullName': 'Marco DI MICHIEL',
    'name': 'Marco',
    'surname': 'Di Michiel',
    'rank': 3,
    'pid': None}],
  'openAccessColor': None,
  'publiclyFunded': None,
  'type': 'dataset',
  'language': {'code': 'eng', 'label': 'English'},
  'countries': None,
  'subjects': [{'subject': {'scheme': 'keyword', 'value': 'PaN'},
    'provenance': None},
   {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
    'provenance': None},
   {'subject': {'scheme': 'keyword', 'value': 'IH-MA-241'},
    'provenance': None},
   {'subject': {'scheme': 'keyword', 'value': 'ID15A'}, 'provenance': None}],
  'mainTitle': 'IHR',
  'subTitle': None,
  'descri

In [26]:
results[1][0]

{'authors': [{'fullName': 'Mustafa ZENO',
   'name': 'Mustafa',
   'surname': 'Zeno',
   'rank': 1,
   'pid': None},
  {'fullName': 'Jonatan VERING',
   'name': 'Jonatan',
   'surname': 'Vering',
   'rank': 2,
   'pid': None},
  {'fullName': 'Marius AMANN',
   'name': 'Marius',
   'surname': 'Amann',
   'rank': 3,
   'pid': None},
  {'fullName': 'Katharina PARISON',
   'name': 'Katharina',
   'surname': 'Parison',
   'rank': 4,
   'pid': None},
  {'fullName': 'Daniel WOHLWEND',
   'name': 'Daniel',
   'surname': 'Wohlwend',
   'rank': 5,
   'pid': None},
  {'fullName': 'Guenter FRITZ',
   'name': 'Guenter',
   'surname': 'Fritz',
   'rank': 6,
   'pid': None},
  {'fullName': 'Florian FRIEDRICH',
   'name': 'Florian',
   'surname': 'Friedrich',
   'rank': 7,
   'pid': None},
  {'fullName': 'Andrew MCCARTHY',
   'name': 'Andrew',
   'surname': 'Mccarthy',
   'rank': 8,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-4478-0136'},
    'provenance': None}}],
 'openAccessCol

In [27]:
# standardised answers, they always have Subjects although it's mostly empty
# extract them, see if any are non - empty

result_subjects = []
for rez in results:
    if rez == []:
        result_subjects.append([])
    else:
        subj = rez[0]['subjects']
        result_subjects.append(subj)

In [28]:
result_subjects

[[{'subject': {'scheme': 'keyword', 'value': 'ID23-2'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Macromolecular Crystallography'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2658'}, 'provenance': None}],
 [{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2446'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID30B'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Macromolecular Crystallography'},
   'provenance': None}],
 [{'subject': {'scheme': 'keyword', 'value': 'id23-2'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-1998'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Macromolecular Crystallography'},
   'provenance': None}],
 [{'subject': {'scheme': 'keyword', 'value': 

In [37]:
result_subjects[20]

[{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'x-ray probe'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'A08-1-1130'}, 'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'BM08'}, 'provenance': None}]

In [38]:
ix = ix_has_doi[20]
record_raw_metadata[ix]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2041032330</identifier>\n    <datestamp>2025-04-14T07:57:28.289Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2041032330</identifier><creators><creator><creatorName nameType="Personal">Birte RIECHERS</creatorName><givenName>Birte</givenName><familyName>Riechers</familyName><nameIdentifier nameIdentifierScheme="ORCID">

In [31]:
openaire_doi_get_info("10.15151/ESRF-ES-1101291159")

DEBUG:    2026-05-21 13:53:44 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 13:53:44 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1101291159 HTTP/1.1" 200 None


[{'authors': [{'fullName': 'Wanner, Adrian',
    'name': 'Adrian',
    'surname': 'Wanner',
    'rank': 1,
    'pid': None}],
  'openAccessColor': None,
  'publiclyFunded': None,
  'type': 'dataset',
  'language': {'code': 'eng', 'label': 'English'},
  'countries': None,
  'subjects': [{'subject': {'scheme': 'keyword', 'value': 'Life Sciences'},
    'provenance': None},
   {'subject': {'scheme': 'keyword', 'value': 'LS-3193'}, 'provenance': None},
   {'subject': {'scheme': 'keyword', 'value': 'ID16A'}, 'provenance': None}],
  'mainTitle': 'Structure and function of neuronal circuits underlying working memory-related sequential activity',
  'subTitle': None,
  'descriptions': ['Working memory is thought to be mediated by stereotypic neuronal activity patterns in the brain such as neural sequences, in which subsets of neurons are sequentially active in a stereotypical temporal order. Neuronal interactions are fundamentally constrained by the synaptic wiring diagram, but the connectivity 

In [32]:
len([result for result in result_subjects if result != []])

142

In [33]:
len(ix_has_doi)
67

67

In [34]:
from utils.enrichment_methods import openaire_doi_get_info, _opeanaire_get_subjects

res = openaire_doi_get_info("10.15151/ESRF-ES-632216926")
print(res)
res_subjects = _opeanaire_get_subjects(res)
print(res_subjects)

DEBUG:    2026-05-21 13:53:44 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 13:53:44 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-632216926 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Amrita PATEL', 'name': 'Amrita', 'surname': 'Patel', 'rank': 1, 'pid': None}, {'fullName': 'Klaus FUTTERER', 'name': 'Klaus', 'surname': 'Futterer', 'rank': 2, 'pid': None}, {'fullName': 'Didier NURIZZO', 'name': 'Didier', 'surname': 'Nurizzo', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-7367-5098'}, 'provenance': None}}, {'fullName': 'Daniel PANNE', 'name': 'Daniel', 'surname': 'Panne', 'rank': 4, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-9158-5507'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {

In [ ]:
# Mangled, fake DOI sure to get no results:

res = openaire_doi_get_info("123456FALSE")
print(res)
res_subjects = _opeanaire_get_subjects(res)
print(res_subjects)

DEBUG:    2026-05-21 13:53:44 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-21 13:53:45 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=123456FALSE HTTP/1.1" 200 None
{}
[]


# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword